# Portfolio Optimization Using Modern Portfolio Theory
This notebook demonstrates how to use Modern Portfolio Theory (MPT) to optimize a portfolio of stocks. We'll compute expected returns, volatility, Sharpe ratios, and plot the efficient frontier.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
%matplotlib inline


## Load Historical Stock Price Data

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
data = yf.download(tickers, start='2020-01-01', end='2024-01-01')['Adj Close']
returns = data.pct_change().dropna()
mean_returns = returns.mean()
cov_matrix = returns.cov()


## Portfolio Return, Volatility, and Sharpe Ratio Calculations

In [ ]:
def portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.dot(weights, mean_returns) * 252
    std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
    sharpe = returns / std
    return returns, std, sharpe


## Optimize Portfolio for Maximum Sharpe Ratio

In [ ]:
def negative_sharpe(weights, mean_returns, cov_matrix):
    return -portfolio_performance(weights, mean_returns, cov_matrix)[2]

constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0, 1) for _ in range(len(tickers)))
init_guess = [1/len(tickers)] * len(tickers)

optimal = minimize(negative_sharpe, init_guess, args=(mean_returns, cov_matrix), method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = optimal.x
opt_returns, opt_volatility, opt_sharpe = portfolio_performance(opt_weights, mean_returns, cov_matrix)
opt_weights, opt_returns, opt_volatility, opt_sharpe


## Plot the Efficient Frontier

In [ ]:
def simulate_portfolios(num_portfolios, mean_returns, cov_matrix):
    results = np.zeros((3, num_portfolios))
    weights_record = []

    for i in range(num_portfolios):
        weights = np.random.random(len(tickers))
        weights /= np.sum(weights)
        weights_record.append(weights)
        portfolio_return, portfolio_std_dev, portfolio_sharpe = portfolio_performance(weights, mean_returns, cov_matrix)
        results[0,i] = portfolio_return
        results[1,i] = portfolio_std_dev
        results[2,i] = portfolio_sharpe

    return results, weights_record

results, weights_record = simulate_portfolios(10000, mean_returns, cov_matrix)

plt.figure(figsize=(10,6))
plt.scatter(results[1,:], results[0,:], c=results[2,:], cmap='viridis')
plt.xlabel('Volatility')
plt.ylabel('Return')
plt.colorbar(label='Sharpe Ratio')
plt.scatter(opt_volatility, opt_returns, marker='*', color='r', s=200, label='Optimal Portfolio')
plt.legend()
plt.title('Efficient Frontier with Optimal Portfolio')
plt.show()
